In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import string


from sklearn.model_selection import train_test_split

In [ ]:
# Read the TSV files into the DataFrame
#Data Source Hasso Plattner Institut 
# NDPL - Non-duplicates
# DPL - Duplicates
# ncvoters - a snap shot of the snapshot: VR_Snapshot_20181106 
df_ncvoters = pd.read_csv('.../ncvoters.tsv', sep='\t')
DPL = pd.read_csv('...ncvoters_DPL.tsv', sep='\t')
NDPL = pd.read_csv('.../ncvoters_NDPL.tsv', sep='\t')

/var/folders/4z/15wr09ws2bv_y9y731bmm5540000gn/T/ipykernel_54136/1350745065.py:6: DtypeWarning: Columns (59,65,89) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ncvoters = pd.read_csv('/Users/noimotbakare/Dropbox/Mac/Downloads/ncvoters.tsv', sep='\t')


# Variable Selection    

In [3]:
# Preprocessing for our fragemented ID analysis 
# Selecting identifyer variables not related to voting
df_ncvoters_frag_ID = df_ncvoters[[
#Stable Identifiers
#Only to be used for labeling /evaluation only 
# / not as a model feature
    'id', 'ncid', 'voter_reg_num', 
# Primary Model features - Strongest features, Strong entropy, Essential for matching
    #many variable such as name prefx and sufx are sparse we can either use none for missing or 0/1
    'first_name', 'midl_name', 'last_name', 'name_sufx_cd',
    #other varaiabes
# Adress Similarity features - Address is the second strongest identity anchor, 
    # Street name especially high discriminative signal # Unit numbers distinguish household
    #many variable such as unit designator are sparse we can either use none for missing or 0/1
    'house_num', 'street_name', 'street_dir', 'street_type_cd', 'unit_designator', 'unit_num', 'zip_code', 'res_city_desc',
#other varaiabes
# Demographic agreement indicators - Moderate/Supporting Features 
    # these will help us reduce false matches # they are agreement indicators, low-weight similarity features
    #age group rather than age because grouped/bin age is more stable
    'age', 'age_group', 'sex', 'race_code', 'race_desc', 'ethnic_code', 'ethnic_desc', 'birth_place',
# #other varaiabes 
 'phone_num','area_cd'   
 ]]
# print("\nSelected variables 'A' and 'C':")
print(df_ncvoters_frag_ID)

             id      ncid  voter_reg_num first_name  midl_name   last_name  \
0           277  aa173006        9132930  domenique       anna   gramolini   
1           566  aa175613        9136375    deborah                  wells   
2          1369  aa188072        9155533       evan  nathaniel     workman   
3          1567  by580879        9157283     alyssa   danielle      marion   
4          1583   ax51253        9157299    lacosta       gail  williamson   
...         ...       ...            ...        ...        ...         ...   
14178  14181892   er19108          19163     jerome     wilson      crouse   
14179  14182849  bn333629          71381      james    richard      dayton   
14180  14183213   en51685          77917       noah    matthew      pardue   
14181  14183530   er32617          62638     krista    stanley  williamson   
14182  14183738   er30307          60328      chara       lane    o connor   

      name_sufx_cd  house_num           street_name street_dir 

# Merging Data

In [4]:
# Adding labels to DPL and NDPL
DPL['label'] = 1 
NDPL['label'] = 0


print(DPL.head(10))
print(NDPL.head(10))

        id1      id2  label
0  11401396   511079      1
1   3569312  9442163      1
2   3569312  9522852      1
3   3569312  9549525      1
4   9442163  9522852      1
5   9442163  9549525      1
6   9522852  9549525      1
7  12658910  3213055      1
8  12658910  3174121      1
9   3213055  3174121      1
        id1       id2  participation  label
0  11826037   3981077       0.428571      0
1  11914983   2089120       0.428571      0
2  10063200  14074062       0.428571      0
3  13740661  13779277       0.428571      0
4  12909240  12927497       0.428571      0
5  12496955  12523473       0.428571      0
6  10378846  10400866       0.428571      0
7  13145288   9989193       0.285714      0
8   9760941   9976890       0.285714      0
9   1027663    997075       0.285714      0


In [5]:
#Merge Duplicate and Non Duplicate pairs
pairs = pd.concat([DPL, NDPL])

print(pairs.head(20))
print(pairs.tail(20))


         id1       id2  label  participation
0   11401396    511079      1            NaN
1    3569312   9442163      1            NaN
2    3569312   9522852      1            NaN
3    3569312   9549525      1            NaN
4    9442163   9522852      1            NaN
5    9442163   9549525      1            NaN
6    9522852   9549525      1            NaN
7   12658910   3213055      1            NaN
8   12658910   3174121      1            NaN
9    3213055   3174121      1            NaN
10   1033927   9114975      1            NaN
11   8655919    847046      1            NaN
12   3462484  10261176      1            NaN
13    383933   7826253      1            NaN
14   7092750   7022921      1            NaN
15   1658211   6300974      1            NaN
16   9167292    377059      1            NaN
17  13224692   3594362      1            NaN
18    222684   3714715      1            NaN
19    299592   9279844      1            NaN
            id1       id2  label  participation
98122  

In [6]:
# merging ncvoters on DPL and NDPL
pairs = pairs.merge(
    df_ncvoters_frag_ID,
    left_on="id1",
    right_on="id",
    how="left"
)
print(pairs.info())
print(pairs.head(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 107961 entries, 0 to 107960
Data columns (total 29 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   id1              107961 non-null  int64  
 1   id2              107961 non-null  int64  
 2   label            107961 non-null  int64  
 3   participation    98142 non-null   float64
 4   id               107961 non-null  int64  
 5   ncid             107961 non-null  object 
 6   voter_reg_num    107961 non-null  int64  
 7   first_name       107961 non-null  object 
 8   midl_name        107961 non-null  object 
 9   last_name        107961 non-null  object 
 10  name_sufx_cd     107961 non-null  object 
 11  house_num        107961 non-null  int64  
 12  street_name      107961 non-null  object 
 13  street_dir       107961 non-null  object 
 14  street_type_cd   107961 non-null  object 
 15  unit_designator  107961 non-null  object 
 16  unit_num         107961 non-null  obje

In [7]:
#merging id2 
pairs = pairs.merge(
    df_ncvoters_frag_ID,
    left_on="id2",
    right_on="id",
    how="left",
    suffixes=("_1", "_2")
)

print(pairs.info())
print(pairs.head(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 107961 entries, 0 to 107960
Data columns (total 54 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id1                107961 non-null  int64  
 1   id2                107961 non-null  int64  
 2   label              107961 non-null  int64  
 3   participation      98142 non-null   float64
 4   id_1               107961 non-null  int64  
 5   ncid_1             107961 non-null  object 
 6   voter_reg_num_1    107961 non-null  int64  
 7   first_name_1       107961 non-null  object 
 8   midl_name_1        107961 non-null  object 
 9   last_name_1        107961 non-null  object 
 10  name_sufx_cd_1     107961 non-null  object 
 11  house_num_1        107961 non-null  int64  
 12  street_name_1      107961 non-null  object 
 13  street_dir_1       107961 non-null  object 
 14  street_type_cd_1   107961 non-null  object 
 15  unit_designator_1  107961 non-null  object 
 16  un

# Group based entity disjoint - Train Test Val Split 

In [8]:
import networkx as nx
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import train_test_split 

# only duplicates
dup_pairs = pairs[pairs["label"] == 1]

G = nx.Graph()

# add edges
G.add_edges_from(zip(dup_pairs["id1"], dup_pairs["id2"]))

# connected components = entity clusters
components = list(nx.connected_components(G))

In [9]:
# Adding ids back in data 

all_ids = set(df_ncvoters_frag_ID["id"])
ids_in_graph = set(G.nodes())

singletons = all_ids - ids_in_graph

for s in singletons:
    components.append({s})

In [10]:
# Split

train_groups, temp_groups = train_test_split(
    components,
    test_size=0.3,
    random_state=42
)

val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.5,
    random_state=42
)

In [11]:
#Converting groups to ids 
train_ids = set().union(*train_groups)
val_ids   = set().union(*val_groups)
test_ids  = set().union(*test_groups)

In [12]:
#bringing dups and non dups together

train_pairs = pairs[pairs["id1"].isin(train_ids) & pairs["id2"].isin(train_ids)]

val_pairs = pairs[pairs["id1"].isin(val_ids) & pairs["id2"].isin(val_ids)]

test_pairs = pairs[pairs["id1"].isin(test_ids) & pairs["id2"].isin(test_ids)]

In [13]:
print("Train:", len(train_pairs))
print("Val:", len(val_pairs))
print("Test:", len(test_pairs))

Train: 54677
Val: 3573
Test: 3870


In [14]:
print(df_ncvoters_frag_ID.columns)
print(pairs.columns)

print(train_pairs.columns)
print(val_pairs.columns)
print(test_pairs.columns)

Index(['id', 'ncid', 'voter_reg_num', 'first_name', 'midl_name', 'last_name',
       'name_sufx_cd', 'house_num', 'street_name', 'street_dir',
       'street_type_cd', 'unit_designator', 'unit_num', 'zip_code',
       'res_city_desc', 'age', 'age_group', 'sex', 'race_code', 'race_desc',
       'ethnic_code', 'ethnic_desc', 'birth_place', 'phone_num', 'area_cd'],
      dtype='object')
Index(['id1', 'id2', 'label', 'participation', 'id_1', 'ncid_1',
       'voter_reg_num_1', 'first_name_1', 'midl_name_1', 'last_name_1',
       'name_sufx_cd_1', 'house_num_1', 'street_name_1', 'street_dir_1',
       'street_type_cd_1', 'unit_designator_1', 'unit_num_1', 'zip_code_1',
       'res_city_desc_1', 'age_1', 'age_group_1', 'sex_1', 'race_code_1',
       'race_desc_1', 'ethnic_code_1', 'ethnic_desc_1', 'birth_place_1',
       'phone_num_1', 'area_cd_1', 'id_2', 'ncid_2', 'voter_reg_num_2',
       'first_name_2', 'midl_name_2', 'last_name_2', 'name_sufx_cd_2',
       'house_num_2', 'street_name_2'

# Augmentation and Hard Negative Creation



 Augmented positives
 +  typos
  + nicknames
  + address abbreviation

 Hard negatives
  + same household
  + same same lastname
  + same firstname

Medium negative 
  + same first name negative  

In [15]:
#Creating a new variable id_to_clusters to be used in augmented training cell - #4 household negatives 
id_to_cluster = {}
for cluster_id, component in enumerate(components):
    for voter_id in component:
        id_to_cluster[voter_id] = cluster_id

# singletons
next_id = len(components)
for v_id in df_ncvoters_frag_ID["id"]:
    if v_id not in id_to_cluster:
        id_to_cluster[v_id] = next_id
        next_id += 1

print(f"Total IDs mapped: {len(id_to_cluster)}")

Total IDs mapped: 14183


In [16]:
# 1. Utilities (names + street logic)

import random

nickname_dict = {
    "william": ["bill","billy","will"],
    "robert": ["bob","bobby","rob"],
    "james": ["jim","jimmy"],
    "john": ["jack","johnny"],
    "elizabeth": ["liz","beth","lizzy"],
    "margaret": ["maggie","meg","peggy"],
    "katherine": ["kate","kathy"],
    "sarah": ["sara"]
}

street_expand = {
    "st":"street",
    "rd":"road",
    "ave":"avenue",
    "dr":"drive",
    "ln":"lane",
    "blvd":"boulevard"
}

def nickname(name):
    n = str(name).lower()
    if n in nickname_dict:
        return random.choice(nickname_dict[n])
    for k,v in nickname_dict.items():
        if n in v:
            return k
    return name


def typo(name):
    name=list(str(name))
    if len(name)<3:
        return "".join(name)
    i=random.randint(0,len(name)-2)
    name[i],name[i+1]=name[i+1],name[i]
    return "".join(name)


def expand_street(st):
    s=str(st).lower()
    return street_expand.get(s,s)

    

In [17]:

# 2. Duplicate augmentation (label = 1) - This creates multi-field realistic duplicates.

def augment_duplicates(train_pairs):
    augmented = []
    for _, row in train_pairs.iterrows():
        if row["label"] != 1:
            continue
        r = row.copy()
        side1_changed = False
        side2_changed = False

        if random.random() < 0.4:
            if random.random() < 0.5:
                r["first_name_1"] = typo(r["first_name_1"])
                side1_changed = True
            else:
                r["first_name_2"] = typo(r["first_name_2"])
                side2_changed = True

        if random.random() < 0.25:
            if not side1_changed and random.random() < 0.5:
                r["first_name_1"] = nickname(r["first_name_1"])
                side1_changed = True
            elif not side2_changed:
                r["first_name_2"] = nickname(r["first_name_2"])
                side2_changed = True

        if random.random() < 0.25:
            if not side1_changed:
                r["street_type_cd_1"] = expand_street(r["street_type_cd_1"])
            elif not side2_changed:
                r["street_type_cd_2"] = expand_street(r["street_type_cd_2"])

        augmented.append(r)
    return pd.DataFrame(augmented)

#aug_dups2 = augment_duplicates(train_pairs)
aug_dups2 = augment_duplicates(train_pairs)
print(aug_dups2.shape)
print(aug_dups2["first_name_1"].head())
print(train_pairs[train_pairs["label"]==1]["first_name_1"].head())

(6826, 54)
0       reic
1    semaje 
2    semaj e
3    seamje 
4     samaje
Name: first_name_1, dtype: object
0       eric
1    semaje 
2    semaje 
3    semaje 
4     samaje
Name: first_name_1, dtype: object


In [18]:
#3 adding steert_ type expansion in both direction 

def street_type_negatives(train_pairs, n=2000):
    pairs = []
    for _, row in train_pairs.sample(len(train_pairs)).iterrows():
        if row["label"] == 1:
            continue
        # skip if both street types are empty
        if row["street_type_cd_1"] == "" and row["street_type_cd_2"] == "":
            continue
        r = row.copy()
        if random.random() < 0.5:
            r["street_type_cd_1"] = expand_street(r["street_type_cd_1"])
        else:
            r["street_type_cd_2"] = expand_street(r["street_type_cd_2"])
        if r["street_type_cd_1"] != row["street_type_cd_1"] or \
           r["street_type_cd_2"] != row["street_type_cd_2"]:
            r["label"] = 0
            pairs.append(r)
        if len(pairs) >= n:
            break
    return pd.DataFrame(pairs)

In [19]:
#5 Nickname negatives 
#this simply adds nickname negatives by swapping some first names to nick names in the NDPL

def nickname_negatives(train_pairs,n=2000):

    pairs=[]

    for _,row in train_pairs.sample(len(train_pairs)).iterrows():
        if row["label"]==1: 
            continue

        r=row.copy()

        r["first_name_2"]=nickname(r["first_name_2"])

        if r["first_name_2"]!=row["first_name_2"]:

            r["label"]=0
            pairs.append(r)

        if len(pairs)>=n:
            break

    return pd.DataFrame(pairs)

Only kept the hard negatives and augmentations that made a difference. 

Applying hard negatives and augmentations to training set ONLY. 

In [20]:
# augmented training set 
aug_dups = augment_duplicates(train_pairs)

# hard 
street_neg = street_type_negatives(train_pairs)
#household_neg = household_negatives(train_pairs, id_to_cluster)
nickname_neg = nickname_negatives(train_pairs)

train_pairs_aug = pd.concat(
    [train_pairs,
     aug_dups, 
     street_neg,
    #household_neg, 
     nickname_neg],
    ignore_index=True
)

Checking that all augmentations and hard negatives changes worked on TRAINING SET 

In [21]:
# 1. Typo check - first_name should differ from original
typo_changes = aug_dups2[aug_dups2["first_name_1"] != train_pairs.loc[aug_dups2.index, "first_name_1"].values]
print(f"Typo applied to first_name_1: {len(typo_changes)}")

typo_changes2 = aug_dups2[aug_dups2["first_name_2"] != train_pairs.loc[aug_dups2.index, "first_name_2"].values]
print(f"Typo applied to first_name_2: {len(typo_changes2)}")

# 2. Nickname check - first_name should be in nickname dict values
nickname_changes1 = aug_dups2[aug_dups2["first_name_1"].str.lower().isin(
    [n for names in nickname_dict.values() for n in names] + list(nickname_dict.keys())
)]
print(f"Nickname in first_name_1: {len(nickname_changes1)}")

nickname_changes2 = aug_dups2[aug_dups2["first_name_2"].str.lower().isin(
    [n for names in nickname_dict.values() for n in names] + list(nickname_dict.keys())
)]
print(f"Nickname in first_name_2: {len(nickname_changes2)}")

# 3. Street expansion check - should see full words not abbreviations
expanded1 = aug_dups2[aug_dups2["street_type_cd_1"].isin(street_expand.values())]
print(f"Street expanded in street_type_cd_1: {len(expanded1)}")

expanded2 = aug_dups2[aug_dups2["street_type_cd_2"].isin(street_expand.values())]
print(f"Street expanded in street_type_cd_2: {len(expanded2)}")

# 4. Nickname negatives check
nick_negs = train_pairs_aug[
    (train_pairs_aug["label"] == 0) &
    (train_pairs_aug["first_name_2"].str.lower().isin(
        [n for names in nickname_dict.values() for n in names]
    ))
]
print(f"Nickname negatives: {len(nick_negs)}")

# 5. Street type negatives check
street_negs = train_pairs_aug[
    (train_pairs_aug["label"] == 0) &
    (train_pairs_aug["street_type_cd_1"].isin(street_expand.values()) |
     train_pairs_aug["street_type_cd_2"].isin(street_expand.values()))
]
print(f"Street type negatives: {len(street_negs)}")

# # 6. Household negatives check
# household_negs = train_pairs_aug[
#     (train_pairs_aug["label"] == 0) &
#     (train_pairs_aug["house_num_1"] == train_pairs_aug["house_num_2"]) &
#     (train_pairs_aug["street_name_1"] == train_pairs_aug["street_name_2"])
# ]
# print(f"Household negatives: {len(household_negs)}")

# 7. Overall label distribution
counts = train_pairs_aug["label"].value_counts()
pcts = train_pairs_aug["label"].value_counts(normalize=True) * 100
print(pd.DataFrame({"count": counts, "percentage": pcts.round(2)}))

Typo applied to first_name_1: 1349
Typo applied to first_name_2: 1364
Nickname in first_name_1: 447
Nickname in first_name_2: 435
Street expanded in street_type_cd_1: 926
Street expanded in street_type_cd_2: 289
Nickname negatives: 2328
Street type negatives: 2000
       count  percentage
label                   
0      51851       79.16
1      13652       20.84


In [22]:
# viewing augmentation results 
cols=[
"first_name_1","first_name_2",
"last_name_1","last_name_2",
"house_num_1","house_num_2",
"street_name_1","street_name_2",
"street_type_cd_1","street_type_cd_2",
"label" #, "age"
]

train_pairs_aug[cols].sample(40)

,first_name_1,first_name_2,last_name_1,last_name_2,house_num_1,house_num_2,street_name_1,street_name_2,street_type_cd_1,street_type_cd_2,label
34585,irma,matthew,cullipher,pike,3704,58,summer,bear creek,pl,rd,0
27201,gay,anna,nunn,thompson,7064,6,kelly coltrane,pond,dr,rd,0
19502,tanisha,dean,alston,pittman,3706,2512,meriwether,weddington,dr,ave,0
12197,judy,betty,poole,bennett,719,4251,craddock,pine hollow,st,dr,0
63106,torrey,laura,miller,luther,144,126,fairhaven,sutton,lane,pl,0
55996,hsannon,shannon,michalka,trim,4996,70,rolling farm,juno,rd,dr,1
10474,david,heather,legrand,aiken,1016,411,hunter valley,tree,rd,ct,0
23096,matthew,kimberly,edge,counts,2014,9909,paddington,laurel lake,dr,ln,0
45743,jeremiah,william,rickman,vannoy,1218,153,birch,fiddlers ridge,st,,0
37865,areial,maria,arnold,sheridan,120,1,boat landing,duke university west campus,dr,,0


In [23]:
#Checking for nonesensical dups
train_pairs[train_pairs.label==1][
["first_name_1","first_name_2","last_name_1","last_name_2"]
].sample(40)

,first_name_1,first_name_2,last_name_1,last_name_2
847,angelon,angelon,smith,gore
7191,stephanie,stephanie,tyson,tyson
3351,mercedes,mercedes,cruz,cruz
2164,vaughn,vaughn,king,king
2433,angela,angela,gulledge,gulledge
224,mark,mark,caccio,caccio
7704,jennifer,jennifer,hudda,collier
3217,muhammad,muhammad,khan,khan
6317,catherine,catherine,bullock,soumano
2390,joseph,joseph,espinosa,espinosa


In [24]:
# Check that all columns are present in augmented data
print(aug_dups.columns.tolist())

# Spot check - view all fields for a few augmented records
aug_dups.sample(5).T  # .T transposes to see all fields vertically

['id1', 'id2', 'label', 'participation', 'id_1', 'ncid_1', 'voter_reg_num_1', 'first_name_1', 'midl_name_1', 'last_name_1', 'name_sufx_cd_1', 'house_num_1', 'street_name_1', 'street_dir_1', 'street_type_cd_1', 'unit_designator_1', 'unit_num_1', 'zip_code_1', 'res_city_desc_1', 'age_1', 'age_group_1', 'sex_1', 'race_code_1', 'race_desc_1', 'ethnic_code_1', 'ethnic_desc_1', 'birth_place_1', 'phone_num_1', 'area_cd_1', 'id_2', 'ncid_2', 'voter_reg_num_2', 'first_name_2', 'midl_name_2', 'last_name_2', 'name_sufx_cd_2', 'house_num_2', 'street_name_2', 'street_dir_2', 'street_type_cd_2', 'unit_designator_2', 'unit_num_2', 'zip_code_2', 'res_city_desc_2', 'age_2', 'age_group_2', 'sex_2', 'race_code_2', 'race_desc_2', 'ethnic_code_2', 'ethnic_desc_2', 'birth_place_2', 'phone_num_2', 'area_cd_2']


,168,2201,6519,702,7676
id1,6686660,1059252,7019631,7634039,7993504
id2,2007920,305904,2006029,11618869,6555668
label,1,1,1,1,1
participation,NaN,NaN,NaN,NaN,NaN
id_1,6686660,1059252,7019631,7634039,7993504
ncid_1,cj92875,ak124076,cn31662,cw636255,ch51405
voter_reg_num_1,33042452,30164751,30063971,1300274,1000344751
first_name_1,sara,victoria,tina,hteresa,colin
midl_name_1,kate,shaye jordan,brooks,harper,ewing
last_name_1,hammel,gabriel,culver,mauldin,stewart


# Feature Engineering 

In [25]:
from jellyfish import jaro_winkler_similarity
import jellyfish

def build_features(pairs):
    features = pd.DataFrame()
#First name similarity
    features["first_name_sim"] = pairs.apply(
        lambda r: jaro_winkler_similarity(
            str(r["first_name_1"]).lower(), 
            str(r["first_name_2"]).lower()
        ), axis=1
    )
    
  #  features["first_name_exact"] = (pairs["first_name_1"] == pairs["first_name_2"]).astype(int)


    #Last name similarity
    features["last_name_sim"] = pairs.apply(
        lambda r: jaro_winkler_similarity(
            str(r["last_name_1"]).lower(), 
            str(r["last_name_2"]).lower()
        ), axis=1
    )

   # features["last_name_exact"] = (pairs["last_name_1"] == pairs["last_name_2"]).astype(int)



#demographic features
    # features["age_diff"] = (
    #     pd.to_numeric(pairs["age_1"], errors="coerce") -
    #     pd.to_numeric(pairs["age_2"], errors="coerce")
    # ).abs()

    # features["age_exact"] = (features["age_diff"] == 0).astype(int)
    # features["age_close"] = (features["age_diff"] <= 1).astype(int)
    # features["age_grp_match"] = (pairs["age_group_1"] == pairs["age_group_2"]).astype(int)
    
   # features["sex_match"] = (pairs["sex_1"] == pairs["sex_2"]).astype(int)

#phone features
    # features["area_cd_match"] = (pairs["area_cd_1"] == pairs["area_cd_2"]).astype(int)
 
   # features["phone_num_match"] = (pairs["phone_num_1"] == pairs["phone_num_2"]).astype(int)
    
    #Address features
    features["street_name_sim"] = pairs.apply(
        lambda r: jaro_winkler_similarity(
            str(r["street_name_1"]).lower(),
            str(r["street_name_2"]).lower()
        ), axis=1
    )
    
    #features["house_num_match"] = (pairs["house_num_1"] == pairs["house_num_2"]).astype(int)

    features["zip_match"] = (pairs["zip_code_1"] == pairs["zip_code_2"]).astype(int)
    
    #Street_type_match
    features["street_type_match"] = (pairs["street_type_cd_1"] == pairs["street_type_cd_2"]).astype(int)

    features["city_sim"] = (pairs["res_city_desc_1"] == pairs["res_city_desc_2"]).astype(int)
    
#City similarity 
    features["city_sim"] = pairs.apply(
        lambda r: jaro_winkler_similarity(
            str(r["res_city_desc_1"]).lower(),
            str(r["res_city_desc_2"]).lower()
        ), axis=1
    )
    features["zip_code_sim"] = (pairs["zip_code_1"] == pairs["zip_code_2"]).astype(int)

# hard negative signal 

    features["same_addr_diff_name"] = (
    (pairs["house_num_1"] == pairs["house_num_2"]) &
    (pairs["street_name_1"] == pairs["street_name_2"]) &
    (pairs["last_name_1"] != pairs["last_name_2"])
    ).astype(int)

# phonetic features 
    for col in ["first_name", "last_name"]:

        col1 = pairs[f"{col}_1"].fillna("").str.lower()
        col2 = pairs[f"{col}_2"].fillna("").str.lower()

    # # encodings
    #     features[f"{col}_soundex_1"] = pairs[f"{col}_1"].fillna("").apply(jellyfish.soundex)
    #     features[f"{col}_soundex_2"] = pairs[f"{col}_2"].fillna("").apply(jellyfish.soundex)

    #     features[f"{col}_metaphone_1"] = pairs[f"{col}_1"].fillna("").apply(jellyfish.metaphone)
    #     features[f"{col}_metaphone_2"] = pairs[f"{col}_2"].fillna("").apply(jellyfish.metaphone)

    # # matches
    #     features[f"{col}_soundex_match"] = (
    #         features[f"{col}_soundex_1"] == features[f"{col}_soundex_2"]
    #     ).astype(int)

    #     features[f"{col}_metaphone_match"] = (
    #         features[f"{col}_metaphone_1"] == features[f"{col}_metaphone_2"]
    #     ).astype(int)


    return features.select_dtypes(include=["number"])


- Note that the augmentation and hard negatives are passed through only the training set. 

- the aug and hard negatives are not  passed through the evaluation set (val and test) per ML principle (must represent untouched reality). 

In [26]:
#Training set contains augmentation/hard negatives 
X_train = build_features(train_pairs_aug)

#unaltered 
X_val   = build_features(val_pairs)
X_test  = build_features(test_pairs)


In [27]:
#Training set contains augmentation/hard negatives 
y_train = train_pairs_aug["label"]

#unaltered 
y_val   = val_pairs["label"]
y_test  = test_pairs["label"]

## ML model
* Stage 1 General model trains on full data
* Stage 2 Hard Case Evaluation

Stage 1 

In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_val)

print("Logistic Regression — Full Validation")
print(classification_report(y_val, y_pred_lr))

Logistic Regression — Full Validation
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      2104
           1       0.98      0.97      0.97      1469

    accuracy                           0.98      3573
   macro avg       0.98      0.98      0.98      3573
weighted avg       0.98      0.98      0.98      3573



In [29]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="logloss",
    use_label_encoder=False
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_val)

print("XGBoost — Full Validation")
print(classification_report(y_val, y_pred_xgb))

/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [19:35:15] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1744329043786/work/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBoost — Full Validation
              precision    recall  f1-score   support

           0       0.97      0.98      0.98      2104
           1       0.98      0.95      0.97      1469

    accuracy                           0.97      3573
   macro avg       0.97      0.97      0.97      3573
weighted avg       0.97      0.97      0.97      3573



stage 2 

In [30]:
# # “How well do we recover TRUE matches under difficulty?” recall focused 
# #recall duplicate sensetivity
# hard_val_1 = val_pairs[
#     (val_pairs["label"] == 1) &
#     (val_pairs["last_name_1"] != val_pairs["last_name_2"])
# ]

In [31]:
# stronger evaluation 
# How well do we separate hard duplicates from ALL negatives? 
# classification focused - precision recall trade off
hard_val_2 = val_pairs[
    ((val_pairs["label"] == 1) & 
     (val_pairs["last_name_1"] != val_pairs["last_name_2"])) |
    (val_pairs["label"] == 0)
]

In [32]:
# build features 
#X_hard_1 = build_features(hard_val_1)
#y_hard_1 = hard_val_1["label"]

X_hard_2 = build_features(hard_val_2)
y_hard_2 = hard_val_2["label"]


In [33]:
# #evaluate logistic regression
# y_pred_lr_hard_1 = lr.predict(X_hard_1)

# print("Logistic Regression — Hard Cases")
# print(classification_report(y_hard, y_pred_lr_hard_1))

We care about recall and balance (F1), because accuracy is hiding the fact that 17% of real matches are missed, and missing a duplicate is very bad in our case. 

In [34]:
#evaluate logistic regression
y_pred_lr_hard_2 = lr.predict(X_hard_2)

print("Logistic Regression — Hard Cases")
print(classification_report(y_hard_2, y_pred_lr_hard_2))

Logistic Regression — Hard Cases
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      2104
           1       0.86      0.83      0.84       229

    accuracy                           0.97      2333
   macro avg       0.92      0.91      0.91      2333
weighted avg       0.97      0.97      0.97      2333



Even advanced tree-based models fail to recover a meaningful portion 
of hard duplicates, indicating limitations of feature-based approaches.
* XGBoost does NOT improve hard recall
* This justifies why we need a deep learning model to solve these hard cases 


In [35]:

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)
build_features(hard_val_2)
xgb.fit(X_train, y_train)


#Full Validation
from sklearn.metrics import classification_report

y_pred_xgb = xgb.predict(X_val)

print("XGBoost — Full Validation")
print(classification_report(y_val, y_pred_xgb))

#Hard Case2
hard_val = val_pairs[
    ((val_pairs["label"] == 1) & 
     (val_pairs["last_name_1"] != val_pairs["last_name_2"])) |
    (val_pairs["label"] == 0)
]

hard_idx = hard_val.index

X_val_hard = X_val.loc[hard_idx]
y_val_hard = y_val.loc[hard_idx]

y_pred_hard_xgb = xgb.predict(X_val_hard)

print("XGBoost — Hard Cases")
print(classification_report(y_val_hard, y_pred_hard_xgb))

/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [19:35:16] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1744329043786/work/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBoost — Full Validation
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      2104
           1       0.97      0.97      0.97      1469

    accuracy                           0.98      3573
   macro avg       0.98      0.98      0.98      3573
weighted avg       0.98      0.98      0.98      3573

XGBoost — Hard Cases
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      2104
           1       0.83      0.81      0.82       229

    accuracy                           0.97      2333
   macro avg       0.90      0.90      0.90      2333
weighted avg       0.96      0.97      0.97      2333



In [36]:
feat_imp = pd.Series(xgb.feature_importances_, index=X_train.columns)
print(feat_imp.sort_values(ascending=False).head(10))

first_name_sim         0.583386
last_name_sim          0.266377
same_addr_diff_name    0.104872
zip_code_sim           0.020113
city_sim               0.008785
zip_match              0.008437
street_name_sim        0.004590
street_type_match      0.003440
dtype: float32


Traditional models, including both linear and non-linear approaches,
perform well on clean data but struggle significantly on hard identity cases.

Despite increased model complexity, XGBoost does not improve recall and balance,
suggesting the limitation lies in feature representation.

This motivates the use of deep learning to learn richer representations
of identity beyond handcrafted similarity features.